# Problem 50: Multi-Document Synthesis

**Difficulty:** Medium | **Framework:** CrewAI

**Categories:** rag

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/multi-document-synthesis).

## 1. Install Dependencies

In [ ]:
!pip install -q crewai crewai-tools

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The retriever must return multiple relevant documents, not just top-1
- The LLM must synthesize information from all retrieved documents
- The answer must combine facts that span multiple documents
- No single document should contain the complete answer


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from crewai import Agent, Task, Crew
from crewai.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings()

documents = [
    Document(page_content="Project Alpha started in January 2024 with a budget of $2M."),
    Document(page_content="Project Alpha's team consists of 12 engineers and 3 designers."),
    Document(page_content="Project Alpha delivered its MVP in June 2024, two weeks ahead of schedule."),
    Document(page_content="Project Beta has a budget of $5M and started in March 2024."),
    Document(page_content="Project Alpha received positive client feedback with a satisfaction score of 9.2/10."),
]

vectorstore = FAISS.from_documents(documents, embeddings)

# BUG: Only retrieves top-1 document — misses facts spread across multiple docs
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

@tool
def search_projects(question: str) -> str:
    """Search project documentation to answer a question."""
    docs = retriever.invoke(question)
    return "\n".join([doc.page_content for doc in docs])

analyst = Agent(
    role="Project Analyst",
    goal="Answer questions about projects accurately and completely",
    backstory="You are an expert project analyst.",
    tools=[search_projects],
)

task = Task(
    description="Give me a complete summary of Project Alpha.",
    expected_output="A comprehensive summary of Project Alpha",
    agent=analyst,
)

crew = Crew(agents=[analyst], tasks=[task])
print(crew.kickoff())

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Increase the retriever's top-k parameter to return more than one document
2. Format all retrieved documents in the context so the LLM can see information from each
3. Add a prompt instruction telling the LLM to combine facts from multiple sources into a unified answer


## 7. Evaluation Criteria

Check your solution against these criteria:

- Retrieves multiple documents
- Synthesizes across documents
- Provides complete answers
